In [3]:
import pandas as pd

FEATURE_PATH = r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_features_10k.csv"

# The CSV was saved WITHOUT column headers
df = pd.read_csv(FEATURE_PATH, header=None)

# Add the correct feature names
df.columns = [
    "user",
    "session",
    "window_id",
    "mean_velocity",
    "std_velocity",
    "median_velocity",
    "max_velocity",
    "min_velocity",
    "mean_acceleration",
    "std_acceleration",
    "max_acceleration",
    "mean_direction_change",
    "median_direction_change",
    "total_distance",
    "mean_distance",
    "window_duration",
    "idle_events",
    "idle_time",
    "idle_percentage",
    "pause_count",
    "mean_pause",
    "median_pause",
    "std_pause",
    "pause_95",
    "velocity_entropy",
    "direction_entropy",
    "velocity_cv",
    "smoothness",
    "num_events"
]

print(df.columns.tolist())
print("\nShape:", df.shape)

['user', 'session', 'window_id', 'mean_velocity', 'std_velocity', 'median_velocity', 'max_velocity', 'min_velocity', 'mean_acceleration', 'std_acceleration', 'max_acceleration', 'mean_direction_change', 'median_direction_change', 'total_distance', 'mean_distance', 'window_duration', 'idle_events', 'idle_time', 'idle_percentage', 'pause_count', 'mean_pause', 'median_pause', 'std_pause', 'pause_95', 'velocity_entropy', 'direction_entropy', 'velocity_cv', 'smoothness', 'num_events']

Shape: (10000, 29)


In [13]:
# ============================================================
# PRODUCTIVITY SCORE GENERATION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# LOAD EXTRACTED MOUSE FEATURES
# ------------------------------------------------------------

FEATURE_PATH = r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_features_10k.csv"

# The CSV was saved WITHOUT column headers
df = pd.read_csv(FEATURE_PATH, header=None)

# Add the correct feature names
df.columns = [
    "user",
    "session",
    "window_id",
    "mean_velocity",
    "std_velocity",
    "median_velocity",
    "max_velocity",
    "min_velocity",
    "mean_acceleration",
    "std_acceleration",
    "max_acceleration",
    "mean_direction_change",
    "median_direction_change",
    "total_distance",
    "mean_distance",
    "window_duration",
    "idle_events",
    "idle_time",
    "idle_percentage",
    "pause_count",
    "mean_pause",
    "median_pause",
    "std_pause",
    "pause_95",
    "velocity_entropy",
    "direction_entropy",
    "velocity_cv",
    "smoothness",
    "num_events"
]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


# ============================================================
# 1. SELECT FEATURES FOR PRODUCTIVITY INDEX
# ============================================================

# Higher value is NOT always better, so we handle the
# direction of contribution explicitly below.

engagement_features = [
    "idle_percentage",
    "pause_count",
    "mean_pause"
]

efficiency_features = [
    "mean_velocity",
    "mean_distance",
    "total_distance"
]

consistency_features = [
    "velocity_cv",
    "smoothness"
]


# ============================================================
# 2. ROBUST NORMALIZATION
# ============================================================
# We use percentile-based normalization instead of raw
# min-max because mouse data can contain extreme values.

def robust_normalize(series):
    lower = series.quantile(0.05)
    upper = series.quantile(0.95)

    if upper == lower:
        return pd.Series(0.5, index=series.index)

    normalized = (series - lower) / (upper - lower)

    return normalized.clip(0, 1)


# ------------------------------------------------------------
# NORMALIZE FEATURES
# ------------------------------------------------------------

for feature in engagement_features + efficiency_features + consistency_features:
    df[feature + "_norm"] = robust_normalize(df[feature])


# ============================================================
# 3. CONVERT FEATURES INTO PRODUCTIVITY CONTRIBUTIONS
# ============================================================

# Engagement:
# Less idle time -> higher productivity
# Fewer/shorter pauses -> higher productivity

df["idle_score"] = 1 - df["idle_percentage_norm"]

df["pause_score"] = 1 - df["pause_count_norm"]

df["pause_duration_score"] = 1 - df["mean_pause_norm"]


# ------------------------------------------------------------
# ------------------------------------------------------------
# EFFICIENCY
# ------------------------------------------------------------
# Instead of rewarding raw mouse speed/distance, we measure
# how much interaction occurs relative to the available time.
#
# This gives us:
#   1. Movement rate      -> useful activity per unit time
#   2. Velocity stability -> avoids highly erratic movement
#   3. Smoothness         -> controlled interaction
#
# Therefore, simply moving the mouse more/faster does not
# automatically produce a higher productivity score.

# Movement achieved per second
df["movement_rate"] = (
    df["total_distance"] /
    df["window_duration"].replace(0, np.nan)
)

df["movement_rate"] = (
    df["movement_rate"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Normalize movement rate
df["movement_rate_norm"] = robust_normalize(
    df["movement_rate"]
)

# Higher velocity CV = more irregular movement = lower efficiency
df["velocity_stability_score"] = (
    1 - df["velocity_cv_norm"]
)

# Higher smoothness = better controlled interaction
df["smoothness_score"] = df["smoothness_norm"]


# ============================================================
# EFFICIENCY COMPONENT
# ============================================================

df["efficiency_score"] = (
    0.50 * df["movement_rate_norm"] +
    0.25 * df["velocity_stability_score"] +
    0.25 * df["smoothness_score"]
)
# ------------------------------------------------------------
# Consistency:
# Lower velocity variation -> better consistency
# Higher smoothness -> better consistency

df["velocity_consistency_score"] = 1 - df["velocity_cv_norm"]

df["smoothness_score"] = df["smoothness_norm"]


# ============================================================
# 4. COMPONENT SCORES
# ============================================================

# Engagement = 40%

df["engagement_score"] = (
    0.50 * df["idle_score"] +
    0.25 * df["pause_score"] +
    0.25 * df["pause_duration_score"]
)


# Efficiency = 30%




# Consistency = 30%

df["consistency_score"] = (
    0.50 * df["velocity_consistency_score"] +
    0.50 * df["smoothness_score"]
)


# ============================================================
# 5. FINAL PRODUCTIVITY SCORE (0-100)
# ============================================================

df["productivity_score"] = 100 * (
    0.40 * df["engagement_score"] +
    0.30 * df["efficiency_score"] +
    0.30 * df["consistency_score"]
)


# ============================================================
# 6. CHECK RESULT
# ============================================================

print("\nProductivity Score Statistics:")
print(df["productivity_score"].describe())

print("\nSample Scores:")
print(
    df[
        [
            "user",
            "session",
            "window_id",
            "engagement_score",
            "efficiency_score",
            "consistency_score",
            "productivity_score"
        ]
    ].head(10)
)


# ============================================================
# 7. SAVE PRODUCTIVITY DATASET
# ============================================================

OUTPUT_PATH = r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_productivity_scores.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved to:")
print(OUTPUT_PATH)

Shape: (10000, 29)
Columns: ['user', 'session', 'window_id', 'mean_velocity', 'std_velocity', 'median_velocity', 'max_velocity', 'min_velocity', 'mean_acceleration', 'std_acceleration', 'max_acceleration', 'mean_direction_change', 'median_direction_change', 'total_distance', 'mean_distance', 'window_duration', 'idle_events', 'idle_time', 'idle_percentage', 'pause_count', 'mean_pause', 'median_pause', 'std_pause', 'pause_95', 'velocity_entropy', 'direction_entropy', 'velocity_cv', 'smoothness', 'num_events']

Productivity Score Statistics:
count    10000.000000
mean        54.527090
std         24.021172
min          0.000000
25%         38.251228
50%         55.496162
75%         72.813151
max        100.000000
Name: productivity_score, dtype: float64

Sample Scores:
   user  session  window_id  engagement_score  efficiency_score  \
0     1        0          0          0.228152          0.362110   
1     1        0          1          0.349739          0.385568   
2     1        3     

In [14]:
# ============================================================
# PRODUCTIVITY SCORE VALIDATION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. BASIC CHECK
# ------------------------------------------------------------

print("=" * 60)
print("1. BASIC DATA CHECK")
print("=" * 60)

print("Shape:", df.shape)
print("\nScore statistics:")
print(df["productivity_score"].describe())


# ------------------------------------------------------------
# 2. CHECK COMPONENT SCORE DISTRIBUTIONS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("2. COMPONENT SCORE DISTRIBUTIONS")
print("=" * 60)

component_cols = [
    "engagement_score",
    "efficiency_score",
    "consistency_score"
]

print(df[component_cols].describe())


# ------------------------------------------------------------
# 3. CHECK WHICH FEATURES CORRELATE WITH PRODUCTIVITY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("3. FEATURE CORRELATIONS WITH PRODUCTIVITY SCORE")
print("=" * 60)

feature_cols = [
    "mean_velocity",
    "std_velocity",
    "median_velocity",
    "max_velocity",
    "min_velocity",
    "mean_acceleration",
    "std_acceleration",
    "max_acceleration",
    "mean_direction_change",
    "median_direction_change",
    "total_distance",
    "mean_distance",
    "window_duration",
    "idle_events",
    "idle_time",
    "idle_percentage",
    "pause_count",
    "mean_pause",
    "median_pause",
    "std_pause",
    "pause_95",
    "velocity_entropy",
    "direction_entropy",
    "velocity_cv",
    "smoothness",
    "num_events"
]

correlations = (
    df[feature_cols + ["productivity_score"]]
    .corr(numeric_only=True)["productivity_score"]
    .drop("productivity_score")
    .sort_values(ascending=False)
)

print(correlations.to_string())


# ------------------------------------------------------------
# 4. HIGHEST AND LOWEST PRODUCTIVITY WINDOWS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("4. HIGHEST PRODUCTIVITY WINDOWS")
print("=" * 60)

high_cols = [
    "user",
    "session",
    "window_id",
    "idle_percentage",
    "pause_count",
    "mean_pause",
    "mean_velocity",
    "total_distance",
    "velocity_cv",
    "smoothness",
    "engagement_score",
    "efficiency_score",
    "consistency_score",
    "productivity_score"
]

print(
    df.sort_values("productivity_score", ascending=False)
      [high_cols]
      .head(10)
      .to_string(index=False)
)


print("\n" + "=" * 60)
print("5. LOWEST PRODUCTIVITY WINDOWS")
print("=" * 60)

print(
    df.sort_values("productivity_score", ascending=True)
      [high_cols]
      .head(10)
      .to_string(index=False)
)


# ------------------------------------------------------------
# 6. CHECK WHETHER COMPONENT WEIGHTS ARE WORKING
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("6. COMPONENT CONTRIBUTION")
print("=" * 60)

print("Correlation with final score:")

print(
    df[
        [
            "engagement_score",
            "efficiency_score",
            "consistency_score",
            "productivity_score"
        ]
    ]
    .corr()["productivity_score"]
    .drop("productivity_score")
    .sort_values(ascending=False)
)


# ------------------------------------------------------------
# 7. CHECK SCORE RANGE / INVALID VALUES
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("7. VALIDITY CHECK")
print("=" * 60)

print("Scores below 0:", (df["productivity_score"] < 0).sum())
print("Scores above 100:", (df["productivity_score"] > 100).sum())
print("Missing scores:", df["productivity_score"].isna().sum())

print("\nComponent values outside [0,1]:")

for col in component_cols:
    print(
        col,
        "below 0 =", (df[col] < 0).sum(),
        "| above 1 =", (df[col] > 1).sum()
    )


# ------------------------------------------------------------
# 8. PRODUCTIVITY BANDS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("8. PRODUCTIVITY BANDS")
print("=" * 60)

df["productivity_band"] = pd.cut(
    df["productivity_score"],
    bins=[0, 25, 50, 75, 100],
    labels=["Low", "Moderate", "High", "Very High"],
    include_lowest=True
)

print(
    df["productivity_band"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 9. FINAL INTERPRETATION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

print("""
Interpretation:

1. Score must remain within 0–100.
2. Engagement, efficiency and consistency must remain within 0–1.
3. idle_percentage and pause-related features should generally have
   negative correlation with productivity.
4. smoothness should generally have positive correlation.
5. velocity_cv should generally have negative correlation because
   higher variability represents lower consistency.
6. No single component should completely dominate the final score.
7. The highest-score windows should generally show more active,
   consistent mouse behavior than the lowest-score windows.

Do NOT modify the scoring formula yet.
First inspect the outputs above.
""")

1. BASIC DATA CHECK
Shape: (10000, 49)

Score statistics:
count    10000.000000
mean        54.527090
std         24.021172
min          0.000000
25%         38.251228
50%         55.496162
75%         72.813151
max        100.000000
Name: productivity_score, dtype: float64

2. COMPONENT SCORE DISTRIBUTIONS
       engagement_score  efficiency_score  consistency_score
count      10000.000000      10000.000000       10000.000000
mean           0.507880          0.501008           0.639389
std            0.265243          0.248517           0.284070
min            0.000000          0.000000           0.000000
25%            0.287857          0.334034           0.458443
50%            0.501446          0.507549           0.718232
75%            0.709635          0.674288           0.865088
max            1.000000          1.000000           1.000000

3. FEATURE CORRELATIONS WITH PRODUCTIVITY SCORE
smoothness                 0.850168
velocity_entropy           0.818876
mean_acceleration    

In [15]:
# ============================================================
# 9. USER-LEVEL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("9. USER-LEVEL PRODUCTIVITY VALIDATION")
print("=" * 60)

user_summary = (
    df.groupby("user")
      .agg(
          windows=("productivity_score", "count"),
          mean_score=("productivity_score", "mean"),
          median_score=("productivity_score", "median"),
          std_score=("productivity_score", "std"),
          min_score=("productivity_score", "min"),
          max_score=("productivity_score", "max")
      )
      .sort_values("mean_score")
)

print("\nLowest average users:")
print(user_summary.head(10))

print("\nHighest average users:")
print(user_summary.tail(10))


# ============================================================
# 10. USER SCORE SPREAD
# ============================================================

print("\n" + "=" * 60)
print("USER-LEVEL SCORE SPREAD")
print("=" * 60)

print(
    user_summary["mean_score"].describe()
)


# ============================================================
# 11. CHECK USERS WITH EXTREME AVERAGE SCORES
# ============================================================

print("\n" + "=" * 60)
print("USERS WITH EXTREME AVERAGE SCORES")
print("=" * 60)

print("\nUsers with average score < 25:")
print(
    user_summary[user_summary["mean_score"] < 25]
)

print("\nUsers with average score > 75:")
print(
    user_summary[user_summary["mean_score"] > 75]
)


# ============================================================
# 12. WITHIN-USER VARIABILITY
# ============================================================
# We want users to have different scores across their sessions.
# If std_score is almost zero for everyone, the score is not
# capturing behavioral changes within a user.

print("\n" + "=" * 60)
print("WITHIN-USER VARIABILITY")
print("=" * 60)

print(
    user_summary["std_score"].describe()
)

print("\nUsers with lowest score variability:")
print(
    user_summary
    .sort_values("std_score")
    .head(10)
)

print("\nUsers with highest score variability:")
print(
    user_summary
    .sort_values("std_score", ascending=False)
    .head(10)
)


# ============================================================
# 13. SESSION-LEVEL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("SESSION-LEVEL PRODUCTIVITY")
print("=" * 60)

session_summary = (
    df.groupby(["user", "session"])
      .agg(
          windows=("productivity_score", "count"),
          mean_score=("productivity_score", "mean"),
          median_score=("productivity_score", "median"),
          std_score=("productivity_score", "std"),
          min_score=("productivity_score", "min"),
          max_score=("productivity_score", "max")
      )
      .reset_index()
)

print("\nSession summary:")
print(session_summary.head(10))

print("\nOverall session score distribution:")
print(
    session_summary["mean_score"].describe()
)


# ============================================================
# 14. WITHIN-SESSION VARIABILITY
# ============================================================

print("\n" + "=" * 60)
print("WITHIN-SESSION VARIABILITY")
print("=" * 60)

print(
    session_summary["std_score"].describe()
)


# ============================================================
# 15. SCORE DISTRIBUTION BY USER
# ============================================================

print("\n" + "=" * 60)
print("USER SCORE DISTRIBUTION")
print("=" * 60)

user_band_distribution = pd.crosstab(
    df["user"],
    df["productivity_band"],
    normalize="index"
) * 100

print(
    user_band_distribution.round(2).head(15)
)


# ============================================================
# 16. CHECK WHETHER A USER DOMINATES THE DATA
# ============================================================

print("\n" + "=" * 60)
print("WINDOW DISTRIBUTION ACROSS USERS")
print("=" * 60)

print(
    df["user"]
    .value_counts()
    .describe()
)

print("\nUsers with most windows:")
print(df["user"].value_counts().head(10))

print("\nUsers with fewest windows:")
print(df["user"].value_counts().tail(10))


# ============================================================
# 17. FINAL VALIDATION SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINAL BPI VALIDATION SUMMARY")
print("=" * 60)

print(
    f"Number of users: {df['user'].nunique()}"
)

print(
    f"Number of sessions: "
    f"{df[['user', 'session']].drop_duplicates().shape[0]}"
)

print(
    f"Number of windows: {len(df)}"
)

print(
    f"Overall mean score: "
    f"{df['productivity_score'].mean():.2f}"
)

print(
    f"Overall std: "
    f"{df['productivity_score'].std():.2f}"
)

print(
    f"Mean user score: "
    f"{user_summary['mean_score'].mean():.2f}"
)

print(
    f"Std of user mean scores: "
    f"{user_summary['mean_score'].std():.2f}"
)

print(
    f"Mean within-user std: "
    f"{user_summary['std_score'].mean():.2f}"
)

print("\nValidation complete.")


9. USER-LEVEL PRODUCTIVITY VALIDATION

Lowest average users:
      windows  mean_score  median_score  std_score  min_score   max_score
user                                                                     
1001     2927   37.217159     37.892893  20.160425   0.000000   90.434558
1019      596   44.226264     49.457130  20.824409   0.000000   80.841995
1050      535   57.143998     57.775634  17.466737  18.254194   91.666551
1006     1067   58.010122     56.751956  15.948012  17.610214   98.726453
1012     2937   61.814853     60.663421  25.180683   3.557785  100.000000
1         244   69.252197     66.906407  24.626351  20.545686  100.000000
1010      312   69.574990     71.797400  11.800445  41.444996   93.160256
1048     1192   69.969662     71.842253  13.660873  20.638527   96.029526
1030      190   73.418888     73.634502  11.551863  47.046232   96.421289

Highest average users:
      windows  mean_score  median_score  std_score  min_score   max_score
user                      